Development and application of a successful Res-UNet capable of detecting and classifying cancerous bloodcells using a common blood smear.

In [ ]:
# Dataset building:
# Import images and bounding box data:

from dataset_builder import idb1_dataset

train_dataset = idb1_dataset()
val_dataset = idb1_dataset(train=True)

In [ ]:
# Visualize image and centroid-specified bounding boxes:
import torch
import torchvision.transforms.functional as F
from torchvision.utils import draw_bounding_boxes
from PIL import Image


image, bboxes = dataset[0]
image_with_box = draw_bounding_boxes(image, bboxes, width=3)
image_pil = F.to_pil_image(image_with_box)
image_pil.show()

In [ ]:
# Convert bounding boxes to masks:
def boxes_to_mask(
    boxes: torch.Tensor,
    image_height: int,
    image_width: int,
    device: torch.device | None = None,
) -> torch.Tensor:
    """
    boxes: Tensor of shape (N, 4) with [x1, y1, x2, y2]
    returns: Tensor of shape (1, H, W), float32 in {0,1}
    """
    if device is None:
        device = boxes.device if isinstance(boxes, torch.Tensor) else torch.device("cpu")

    mask = torch.zeros((1, image_height, image_width), dtype=torch.float32, device=device)

    if boxes.numel() == 0:
        return mask

    for box in boxes:
        x1, y1, x2, y2 = box.round().long()

        # Clamp to valid image bounds
        x1 = torch.clamp(x1, 0, image_width - 1)
        x2 = torch.clamp(x2, 0, image_width)
        y1 = torch.clamp(y1, 0, image_height - 1)
        y2 = torch.clamp(y2, 0, image_height)

        if x2 > x1 and y2 > y1:
            mask[:, y1:y2, x1:x2] = 1.0

    return mask

In [ ]:
import os
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models



# ============================================================
# 2. Convert bounding boxes to binary mask
# ============================================================

def boxes_to_mask(
    boxes: torch.Tensor,
    image_height: int,
    image_width: int,
    device: torch.device | None = None,
) -> torch.Tensor:
    """
    boxes: Tensor of shape (N, 4) with [x1, y1, x2, y2]
    returns: Tensor of shape (1, H, W), float32 in {0,1}
    """
    if device is None:
        device = boxes.device if isinstance(boxes, torch.Tensor) else torch.device("cpu")

    mask = torch.zeros((1, image_height, image_width), dtype=torch.float32, device=device)

    if boxes.numel() == 0:
        return mask

    for box in boxes:
        x1, y1, x2, y2 = box.round().long()

        # Clamp to valid image bounds
        x1 = torch.clamp(x1, 0, image_width - 1)
        x2 = torch.clamp(x2, 0, image_width)
        y1 = torch.clamp(y1, 0, image_height - 1)
        y2 = torch.clamp(y2, 0, image_height)

        if x2 > x1 and y2 > y1:
            mask[:, y1:y2, x1:x2] = 1.0

    return mask


# ============================================================
# 3. Custom collate function
#    Boxes have variable length per image, so default collation
#    may fail. We stack images and keep boxes as a list.
# ============================================================

def collate_fn_boxes(batch: List[Tuple[torch.Tensor, torch.Tensor]]):
    images, boxes = zip(*batch)
    images = torch.stack(images, dim=0)   # (B, 3, H, W)
    return images, list(boxes)


# ============================================================
# 4. Simple ResUNet-style model
#    This uses a ResNet encoder and a lightweight decoder.
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = ConvBlock(in_ch // 2 + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)

        # handle any off-by-one spatial differences
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)

        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class ResUNet(nn.Module):
    def __init__(self, out_channels=1, pretrained=True):
        super().__init__()

        backbone = models.resnet34(
            weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        )

        # Encoder
        self.input_block = nn.Sequential(
            backbone.conv1,   # 7x7, stride 2
            backbone.bn1,
            backbone.relu,
        )  # -> 64 channels
        self.maxpool = backbone.maxpool

        self.encoder1 = backbone.layer1   # 64
        self.encoder2 = backbone.layer2   # 128
        self.encoder3 = backbone.layer3   # 256
        self.encoder4 = backbone.layer4   # 512

        # Decoder
        self.center = ConvBlock(512, 512)
        self.up4 = UpBlock(512, 256, 256)
        self.up3 = UpBlock(256, 128, 128)
        self.up2 = UpBlock(128, 64, 64)
        self.up1 = UpBlock(64, 64, 64)

        self.final_up = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, out_channels, kernel_size=1)
        )

    def forward(self, x):
        # Encoder path
        x0 = self.input_block(x)      # (B, 64, H/2, W/2)
        x1 = self.maxpool(x0)         # (B, 64, H/4, W/4)
        x2 = self.encoder1(x1)        # (B, 64, H/4, W/4)
        x3 = self.encoder2(x2)        # (B,128, H/8, W/8)
        x4 = self.encoder3(x3)        # (B,256, H/16, W/16)
        x5 = self.encoder4(x4)        # (B,512, H/32, W/32)

        # Decoder path
        c = self.center(x5)
        d4 = self.up4(c, x4)
        d3 = self.up3(d4, x3)
        d2 = self.up2(d3, x2)
        d1 = self.up1(d2, x0)

        out = self.final_up(d1)

        # ensure same spatial size as input
        if out.shape[-2:] != x.shape[-2:]:
            out = F.interpolate(out, size=x.shape[-2:], mode="bilinear", align_corners=False)

        return out


# ============================================================
# 5. Losses
# ============================================================

def dice_loss_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-6):
    probs = torch.sigmoid(logits)
    probs = probs.contiguous().view(probs.size(0), -1)
    targets = targets.contiguous().view(targets.size(0), -1)

    intersection = (probs * targets).sum(dim=1)
    denom = probs.sum(dim=1) + targets.sum(dim=1)

    dice = (2.0 * intersection + eps) / (denom + eps)
    return 1.0 - dice.mean()


class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=1.0, dice_weight=1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        return (
            self.bce_weight * self.bce(logits, targets)
            + self.dice_weight * dice_loss_from_logits(logits, targets)
        )


# ============================================================
# 6. Training / validation loops
# ============================================================

def build_masks_from_box_list(
    box_list: List[torch.Tensor],
    image_height: int,
    image_width: int,
    device: torch.device,
) -> torch.Tensor:
    masks = [
        boxes_to_mask(boxes.to(device), image_height, image_width, device=device)
        for boxes in box_list
    ]
    return torch.stack(masks, dim=0)   # (B, 1, H, W)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    total = 0

    for images, box_list in loader:
        images = images.to(device)
        B, _, H, W = images.shape

        masks = build_masks_from_box_list(box_list, H, W, device)

        optimizer.zero_grad()
        logits = model(images)                # (B,1,H,W)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * B
        total += B

    return running_loss / max(total, 1)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    total = 0

    for images, box_list in loader:
        images = images.to(device)
        B, _, H, W = images.shape

        masks = build_masks_from_box_list(box_list, H, W, device)

        logits = model(images)
        loss = criterion(logits, masks)

        running_loss += loss.item() * B
        total += B

    return running_loss / max(total, 1)


# ============================================================
# 7. Example main training function
# ============================================================

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --------------------------------------------------------
    # Replace this section with your real dataset tensors.
    # Here we build fake data just to show interface.
    # --------------------------------------------------------
    num_train = 20
    num_val = 6
    H, W = 256, 256

    train_images = [torch.rand(3, H, W) for _ in range(num_train)]
    train_boxes = [
        torch.tensor([[40, 50, 100, 120], [140, 130, 200, 210]], dtype=torch.float32)
        if i % 2 == 0 else
        torch.tensor([[60, 70, 110, 130]], dtype=torch.float32)
        for i in range(num_train)
    ]

    val_images = [torch.rand(3, H, W) for _ in range(num_val)]
    val_boxes = [
        torch.tensor([[30, 40, 80, 95]], dtype=torch.float32)
        for _ in range(num_val)
    ]

    train_loader = DataLoader(
        train_dataset,
        batch_size=4,
        shuffle=True,
        num_workers=0,
        collate_fn=collate_fn_boxes,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=4,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_fn_boxes,
    )

    model = ResUNet(out_channels=1, pretrained=True).to(device)
    criterion = BCEDiceLoss(bce_weight=1.0, dice_weight=1.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

    epochs = 10
    best_val = float("inf")

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), "resunet_allidb1_from_boxes.pt")
            print("  saved best model")

    print("Done.")


if __name__ == "__main__":
    main()
